# OSeMOSYS vs PyPSA — capacity expansion on identical inputs

Both frameworks solve the *same* system: one `tz-osemosys` `model.json`, the same
96 timeslices × 28 years, the same 9 Japan grid regions, the same 16 technologies.
No re-clustering — the OSeMOSYS timeslices **are** the PyPSA snapshots.

This notebook answers three questions from saved artefacts:

1. **Do the two frameworks find the same answer?** (yes — to 0.02 % on generation)
2. **What does it cost to compare the objectives honestly?** (one constant term, and
   two documented discounting conventions)
3. **Why is PyPSA so much slower on the same LP?** (it isn't the answer — it's the
   formulation: a vintage axis that lands on dispatch)

Background reading, in order: `REPORT.md` for the findings, then
`osemosys_vs_pypsa/METHODOLOGY.md` for how the translation works and what was
removed from the OSeMOSYS model to make the comparison fair.

## Running it

Nothing here solves anything — it reads run directories. To use the reference runs
that came with the repo, just run all cells. To point it at your own run:

```bash
python run_benchmark.py artifacts/model.json --outdir runs/mine --build-year-step 5
```

then set `SOLUTIONS_RUN = Path("runs/mine")` and `TIMING_RUN = Path("runs/mine")` below.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from osemosys_vs_pypsa import analysis as A
from osemosys_vs_pypsa.runio import Run

# Two reference runs, because no single original run has both halves:
#   step1-gurobi -- both sides solved to optimality  -> use for results agreement
#   step1-highs  -- the REPORT.md solve-time comparison; PyPSA never finished,
#                   so it has a log and no solved network
# Point both at your own run directory to analyse your own solve.
SOLUTIONS_RUN = Path("artifacts/reference/step1-gurobi")
TIMING_RUN = Path("artifacts/reference/step1-highs")

A.apply_theme()
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

solutions = Run(SOLUTIONS_RUN)
timing = Run(TIMING_RUN)
pd.DataFrame([solutions.describe(), timing.describe()]).set_index("run")

## 1. What was solved

`manifest.json` records the solver, the build-year step and the exact package
versions. Solve times are only comparable across runs on the same versions —
HiGHS in particular changes interior-point behaviour between releases.

In [ ]:
network = solutions.pypsa_network
solution = solutions.osemosys_solution

print(f"PyPSA:    {len(network.snapshots):,} snapshots over "
      f"{len(network.investment_periods)} investment periods")
print(f"          {len(network.generators):,} generators "
      f"({int(network.generators.p_nom_extendable.sum()):,} extendable), "
      f"{len(network.links):,} links")
print(f"OSeMOSYS: {solution.sizes['TECHNOLOGY']} technologies, "
      f"{solution.sizes['REGION']} regions, {solution.sizes['TIMESLICE']} timeslices, "
      f"{solution.sizes['YEAR']} years")

pd.Series(solutions.manifest.get("versions", {}), name="version").to_frame()

## 2. The objective trap

**`TotalDiscountedCost` in `solution.nc` is not the OSeMOSYS LP objective.**

linopy drops the objective's constant term (there is a `TODO` about it in
`tz.osemosys.Model.solve`), so the solver minimises everything *except* fixed O&M
charged on `ResidualCapacity` — while the saved variable includes it. Compare the
saved number against PyPSA's objective and you get a ~58 % gap that is entirely an
accounting artefact.

PyPSA omits the same term for an unrelated reason: the residual fleet is modelled
as **non-extendable** components, and `define_objective` draws its cost terms from
`c.extendables`, so the residual fleet never enters the objective at all.

Both sides exclude it, so subtract it from the OSeMOSYS total and the two LP
objectives are directly comparable.

In [ ]:
objectives = A.objective_table(solution, network)
display(objectives.round(1))

print(f"PyPSA is {objectives.attrs['difference_millions']:>9,.0f} $m higher "
      f"({objectives.attrs['difference_pct']:+.2f} %)")

# The subtraction is exact, not approximate: it reproduces the solver's own number.
logged = timing.osemosys_log.objective
derived = A.compare(solution, network)["osemosys"]["lp_objective"]
print(f"solver log reported {logged:,.1f} $m; derived from solution.nc {derived:,.1f} $m "
      f"(difference {abs(logged - derived):.2e})")

### What the remaining 1.68 % is

Two genuine framework conventions, both documented in `METHODOLOGY.md` — neither is
a translation error:

1. **Annuity vs annuity-due.** PyPSA's `annuity(r, N) = r / (1 − (1+r)^−N)`;
   OSeMOSYS's capital recovery factor is the same divided by `(1+r)`. PyPSA
   therefore charges `(1 + r_idv)×` OSeMOSYS's capital cost — **+3.1 %** at
   `r_idv = 0.031`.
2. **Mid-year vs start-of-year opex discounting.** OSeMOSYS discounts operating
   cost by `(1+r)^−(y−y₀+0.5)`, PyPSA by `(1+r)^−(y−y₀)`, so PyPSA values the same
   opex `(1+r)^0.5` higher — **+1.5 %** at `r = 0.03`.

The same convention split shows up in the constant we just removed: it is
364,282 $m discounted OSeMOSYS-style, 369,706 $m PyPSA-style.

In [ ]:
from osemosys_vs_pypsa.reconcile import residual_fixed_om

pd.Series(
    {
        "OSeMOSYS convention (mid-year)": residual_fixed_om(solution, "osemosys"),
        "PyPSA convention (start-of-year)": residual_fixed_om(solution, "pypsa"),
    },
    name="discounted fixed O&M on residual capacity ($m)",
).to_frame().round(0)

## 3. Do the two frameworks find the same answer?

This is the question that decides whether the solve-time gap is interesting. If the
answers differed, the gap would just mean the models are different. They don't.

In [ ]:
capacity = A.new_capacity_by_technology(solution, network)
generation = A.generation_by_technology(solution, network)

summary = A.agreement_summary(capacity, generation)
print(f"new capacity   OSeMOSYS {summary['capacity_total_osemosys_GW']:>12,.3f} GW")
print(f"               PyPSA    {summary['capacity_total_pypsa_GW']:>12,.3f} GW")
print(f"               worst per-technology difference "
      f"{summary['capacity_max_abs_diff_GW']:.2e} GW")
print()
print(f"generation     OSeMOSYS {summary['generation_total_osemosys_TWh']:>12,.1f} TWh")
print(f"               PyPSA    {summary['generation_total_pypsa_TWh']:>12,.1f} TWh")
print(f"               worst per-technology difference "
      f"{summary['generation_max_abs_diff_TWh']:.2f} TWh "
      f"({summary['generation_max_abs_diff_pct']:.3f} % of the system)")

In [ ]:
display(generation[generation["OSeMOSYS"] > 0].round(2))
display(capacity[capacity["OSeMOSYS"] > 0].round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
A.plot_generation_by_technology(generation, axes[0])
A.plot_generation_difference(generation, axes[1])
fig.tight_layout()

In [ ]:
capacity_by_year = A.new_capacity_by_year(solution, network)

fig, ax = plt.subplots(figsize=(9, 4.4))
A.plot_new_capacity_by_year(capacity_by_year, ax)
fig.tight_layout()

capacity_by_year[capacity_by_year.sum(axis=1) > 0].round(3)

## 4. Where the wall clock goes

Same `model.json`, same solver, same machine. Everything below is read straight out
of the two solver logs by `osemosys_vs_pypsa.solverlog`.

The investment problem is **identical** — 4,592 capacity variables on each side. The
entire gap is dispatch, and it comes from where the build-year axis lands:

* **OSeMOSYS** sums vintages *before* bounding dispatch, so `CAa4` is **one row** per
  (region, tech, timeslice, year) with ~15 capacity terms in it, and `RateOfActivity`
  carries no build-year index.
* **PyPSA** bounds dispatch per component, and a component *is* a vintage. Each
  (tech, region) has ~14.5 live vintages, each with its own `p` variable and a
  two-sided `p ≤ p_max_pu · p_nom` constraint pair per snapshot.

Same feasible set — summing PyPSA's 28 rows per timeslice reproduces the OSeMOSYS row
exactly, because vintages of a technology here share one marginal cost and one
`p_max_pu`. PyPSA pays 14.5 × 2 rows for expressiveness this dataset cannot use.

In [ ]:
lp = A.lp_comparison(timing.osemosys_log, timing.pypsa_log)
lp

Reading that table, the gap decomposes into three multiplicative pieces:

1. **Per-iteration cost, ~5.6×.** Counter-intuitively OSeMOSYS does *more* arithmetic
   per iteration — its aggregated rows are denser. PyPSA loses on memory, not flops:
   a 12 GB working set against 1.4 GB, streamed every iteration. The solve is
   bandwidth-bound.
2. **Iteration count, 1.7×.** Conditioning. PyPSA emits costs in $/MW, so its cost
   range is `[3e1, 2e7]` and HiGHS warns about excessively large costs; OSeMOSYS's
   $m/GW keeps it in `[3e-3, 2e4]`.
3. **Crossover failure — the single biggest item.** With ~14.5 *identical* vintages
   per technology, any split of dispatch across them is equally optimal, so the LP
   optimum is a huge degenerate face. The interior-point method converges to the
   centre of it and crossover must push millions of tied variables to a vertex:
   **6.02 M dual pushes against 180 k** (33×). It ends `imprecise`, HiGHS falls back
   to dual simplex, and the run never finishes.

In [ ]:
for name, log in (("OSeMOSYS", timing.osemosys_log), ("PyPSA", timing.pypsa_log)):
    print(f"{name}: {log.solver}, completed={log.completed}")
    print(f"  cost range        {log.cost_range}")
    print(f"  IPM iterations    {log.ipm_iterations} @ "
          f"{log.seconds_per_ipm_iteration:.1f} s/iteration")
    print(f"  crossover         {log.crossover_status}, "
          f"{log.dual_pushes:,} dual pushes / {log.primal_pushes:,} primal")
    if not log.completed:
        print(f"  log ends at       iteration {log.last_simplex_iteration:,} "
              f"after {log.last_simplex_seconds:,.0f} s, still in simplex cleanup")
    for warning in log.warnings:
        print(f"  warning           {warning}")
    print()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))
A.plot_wall_clock(timing.osemosys_log, timing.pypsa_log, ax)
fig.tight_layout()

### The same comparison on Gurobi

HiGHS never finishes the PyPSA side, which makes the gap a lower bound. Gurobi
finishes both, so it puts a number on it — and note Gurobi solved the OSeMOSYS LP
with dual simplex in 36 s, sidestepping the degenerate-face problem entirely.

In [ ]:
gurobi_lp = A.lp_comparison(solutions.osemosys_log, solutions.pypsa_log)
display(gurobi_lp.loc[["solver", "status", "wall_clock_s", "rows", "cols",
                       "ipm_iterations", "objective"]])

fig, ax = plt.subplots(figsize=(9, 3.2))
A.plot_wall_clock(solutions.osemosys_log, solutions.pypsa_log, ax)
fig.tight_layout()

## 5. Levers, in order of impact

1. **Coarsen the vintage axis.** `--build-year-step 5` cuts PyPSA dispatch variables
   ~4× (6.62 M → 1.70 M) and, more importantly, shrinks the degenerate face that
   kills crossover. `run_benchmark.py` applies the same restriction to the OSeMOSYS
   side — it pins `capacity_additional_max` to zero in the off-step years — so the
   comparison stays fair. Without that, you would be comparing a coarse PyPSA
   against a fine OSeMOSYS.
2. **Skip crossover** (`--no-crossover`) if you don't need a basic solution. The
   interior point was already primal-dual feasible to 1e-9 long before the cleanup
   started.
3. **Scale the objective** — emit $m/GW, or set HiGHS's `user_objective_scale=-5`,
   to fix the conditioning warning and claw back some of the 151-vs-90 iterations.
4. **Not a lever:** PyPSA cannot express OSeMOSYS's vintage-aggregated dispatch.
   `GrossCapacity` has no PyPSA analogue — capacity exists only as an attribute of a
   dispatching component — so the residual ~10× formulation gap is structural.

### Reproducing this yourself

```bash
# the runnable default: 5-yearly investment on both sides, HiGHS
python run_benchmark.py artifacts/model.json --outdir runs/step5 --build-year-step 5

# the headline run: needs Gurobi and ~16 GB; HiGHS will not finish the PyPSA side
python run_benchmark.py artifacts/model.json --outdir runs/step1 \
    --build-year-step 1 --solver gurobi

# regenerate the labelled LP files (0.9 GB + 5.8 GB) to read the rows yourself
python run_benchmark.py artifacts/model.json --outdir runs/lp --write-lp
```

Then set `SOLUTIONS_RUN` and `TIMING_RUN` at the top of this notebook to your run
directory and re-run.